# 02 - Bronze Transformations

## Objetivo
Este notebook tem como objetivo transformar os dados da camada `01_raw` em tabelas estruturadas na camada `02_bronze`, aplicando tratamentos técnicos mínimos para padronização e preparação das bases.

## Escopo da camada bronze
Nesta etapa, serão realizados apenas tratamentos técnicos, tais como:
- padronização de nomes de colunas
- ajuste de tipos de dados
- normalização estrutural de arquivos JSON
- remoção de duplicidades quando aplicável
- padronização técnica de campos categóricos essenciais

## Tabelas processadas
- `lh_nautical.01_raw.vendas_raw`
- `lh_nautical.01_raw.produtos_raw`
- `lh_nautical.01_raw.clientes_raw`
- `lh_nautical.01_raw.custos_importacao_raw`

## Saídas esperadas
- `lh_nautical.02_bronze.vendas_bronze`
- `lh_nautical.02_bronze.produtos_bronze`
- `lh_nautical.02_bronze.clientes_bronze`
- `lh_nautical.02_bronze.custos_importacao_bronze`

## Observação
Regras de negócio, joins entre entidades, métricas analíticas e tabelas finais para consumo serão tratados nas camadas `03_silver` e `04_gold`.

In [0]:
# ======================================================================================
# IMPORTS
# ======================================================================================
# Bibliotecas e funções auxiliares utilizadas nas transformações da camada bronze.
# ======================================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType
import unicodedata

In [0]:
# ======================================================================================
# SESSÃO SPARK
# ======================================================================================


spark = SparkSession.builder.getOrCreate()

## Funções auxiliares

As funções abaixo padronizam nomes de colunas e categorias de produtos, além de facilitar a reutilização de lógica ao longo do notebook.

In [0]:
# ======================================================================================
# FUNÇÕES AUXILIARES
# ======================================================================================

def standardize_column_name(column_name: str) -> str:
    """
    Padroniza nomes de colunas para snake_case simples, em minúsculas.

    Regras aplicadas:
    - remove espaços nas extremidades
    - converte para minúsculas
    - substitui espaços por underscore
    - remove caracteres especiais básicos
    """
    cleaned = column_name.strip().lower()
    cleaned = cleaned.replace(" ", "_")
    cleaned = cleaned.replace("-", "_")
    cleaned = cleaned.replace("/", "_")
    cleaned = cleaned.replace("(", "")
    cleaned = cleaned.replace(")", "")
    cleaned = cleaned.replace(".", "")
    return cleaned


def normalize_text(text: str) -> str:
    """
    Remove acentos e padroniza texto para comparação.
    Retorna string em minúsculas e sem espaços excedentes.
    """
    if text is None:
        return None
    
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    return text


def normalize_category(category: str) -> str:
    """
    Padroniza categorias de produtos para os três valores de negócio esperados:
    - eletronicos
    - propulsao
    - ancoragem

    Mantém None se a categoria estiver ausente.
    """
    if category is None:
        return None

    category_norm = normalize_text(category)

    if "eletr" in category_norm:
        return "eletronicos"
    elif "prop" in category_norm:
        return "propulsao"
    elif "ancor" in category_norm:
        return "ancoragem"
    else:
        return category_norm

In [0]:
# ======================================================================================
# REGISTRO DE UDF
# ======================================================================================
# A função Python de normalização de categoria é registrada como UDF para aplicação
# sobre DataFrames Spark.
# ======================================================================================

normalize_category_udf = F.udf(normalize_category)

In [0]:
# ======================================================================================
# FUNÇÃO DE PADRONIZAÇÃO DE COLUNAS
# ======================================================================================

def standardize_dataframe_columns(df):
    """
    Renomeia todas as colunas do DataFrame para um padrão técnico consistente.
    """
    renamed_columns = [standardize_column_name(col_name) for col_name in df.columns]
    return df.toDF(*renamed_columns)

## 1. Transformação da tabela de produtos

Nesta etapa serão aplicados:
- padronização dos nomes de colunas
- normalização das categorias
- conversão de colunas numéricas
- remoção de duplicidades

In [0]:
# ======================================================================================
# LEITURA - PRODUTOS RAW
# ======================================================================================

df_produtos_raw = spark.table("lh_nautical.01_raw.produtos_raw")
print("Quantidade de registros em produtos_raw:", df_produtos_raw.count())
display(df_produtos_raw.limit(10))

In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - PRODUTOS BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - usar a categoria original disponível no dataset
# - criar a coluna category padronizada
# - limpar campos monetários antes do cast
# - converter campos numéricos
# - remover duplicados
# ======================================================================================

df_produtos_bronze = standardize_dataframe_columns(df_produtos_raw)

# Criação da coluna category padronizada a partir da coluna original actual_category
if "actual_category" in df_produtos_bronze.columns:
    df_produtos_bronze = df_produtos_bronze.withColumn(
        "category",
        normalize_category_udf(F.col("actual_category"))
    )

# Limpeza de campos monetários antes da conversão para double
monetary_columns = ["price", "cost"]

for col_name in monetary_columns:
    if col_name in df_produtos_bronze.columns:
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), r"R\$\s*", "")
        )
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), ",", ".")
        )

# Conversão de tipos numéricos
numeric_cast_map = {
    "code": IntegerType(),
    "price": DoubleType(),
    "cost": DoubleType(),
    "stock": IntegerType()
}

for col_name, col_type in numeric_cast_map.items():
    if col_name in df_produtos_bronze.columns:
        df_produtos_bronze = df_produtos_bronze.withColumn(
            col_name,
            F.col(col_name).cast(col_type)
        )

# Remoção de duplicidades exatas
total_produtos_antes = df_produtos_bronze.count()

df_produtos_bronze = df_produtos_bronze.dropDuplicates()

total_produtos_depois = df_produtos_bronze.count()
duplicados_removidos_produtos = total_produtos_antes - total_produtos_depois

print("Duplicados removidos em produtos:", duplicados_removidos_produtos)

display(df_produtos_bronze.limit(10))

In [0]:
# Conferir categorias inválidas
valid_categories = ["eletronicos", "propulsao", "ancoragem"]

display(
    df_produtos_bronze
    .filter(~F.col("category").isin(valid_categories) | F.col("category").isNull())
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
def normalize_category(category: str) -> str:
    """
    Padroniza categorias de produtos para:
    - eletronicos
    - propulsao
    - ancoragem

    Inclui tratamento de variações com erro de digitação e espaçamento.
    """
    if category is None:
        return None

    category_norm = normalize_text(category)
    category_norm = category_norm.replace(" ", "")

    # Eletrônicos
    if "eletr" in category_norm:
        return "eletronicos"

    # Propulsão
    if "prop" in category_norm:
        return "propulsao"

    # Ancoragem (incluindo erros comuns)
    if (
        "ancor" in category_norm
        or "encor" in category_norm
        or category_norm in [
            "ancoraguem",
            "ancorajm",
            "ancorajem",
            "ancorajen",
            "ancoragen",
            "encoragem",
            "encoragi"
        ]
    ):
        return "ancoragem"

    return None


normalize_category_udf = F.udf(normalize_category, StringType())

In [0]:
df_produtos_bronze = df_produtos_bronze.withColumn(
    "category",
    normalize_category_udf(F.col("actual_category"))
)

In [0]:
display(
    df_produtos_bronze
    .filter(F.col("category").isNull())
    .groupBy("actual_category")
    .count()
    .orderBy(F.desc("count"), F.asc("actual_category"))
)

In [0]:
display(
    df_produtos_bronze
    .groupBy("category")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_produtos_bronze
    .select("actual_category", "category")
    .distinct()
    .orderBy("actual_category")
)

In [0]:
# Renomear colunas
df_produtos_bronze = df_produtos_bronze.withColumnRenamed("code", "product_id")
df_produtos_bronze = df_produtos_bronze.withColumnRenamed("name", "product_name")

In [0]:
# ======================================================================================
# ESCRITA - PRODUTOS BRONZE
# ======================================================================================

df_produtos_bronze.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.produtos_bronze")

In [0]:
display(
    spark.table("lh_nautical.02_bronze.produtos_bronze")
)

## 2. Transformação da tabela de custos de importação

Nesta etapa será realizado o tratamento estrutural do JSON aninhado, transformando o campo `historic_data` em múltiplas linhas.

Saída esperada:
- uma linha por produto, por data de vigência de custo, contendo o valor em dólar

In [0]:
# ======================================================================================
# LEITURA - CUSTOS DE IMPORTAÇÃO RAW
# ======================================================================================

df_importacao_raw = spark.table("lh_nautical.01_raw.custos_importacao_raw")
print("Quantidade de registros em custos_importacao_raw:", df_importacao_raw.count())
display(df_importacao_raw.limit(10))

In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - CUSTOS DE IMPORTAÇÃO BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - explodir o array historic_data
# - selecionar colunas finais da estrutura normalizada
# - converter tipos de dados
# - tratar datas no formato brasileiro (dd/MM/yyyy)
# - remover registros críticos nulos
# ======================================================================================

df_importacao_raw = standardize_dataframe_columns(df_importacao_raw)

# Explode do array histórico para obter uma linha por produto e data de vigência
df_importacao_exploded = df_importacao_raw.withColumn(
    "historic_record",
    F.explode("historic_data")
)

# Seleção e tipagem das colunas finais
df_importacao_bronze = df_importacao_exploded.select(
    F.col("product_id").cast(IntegerType()).alias("product_id"),
    F.col("product_name").alias("product_name"),
    normalize_category_udf(F.col("category")).alias("category"),
    F.to_date(F.col("historic_record.start_date"), "dd/MM/yyyy").alias("start_date"),
    F.col("historic_record.usd_price").cast(DoubleType()).alias("usd_price")
)

# Remoção de registros com campos essenciais nulos
df_importacao_bronze = df_importacao_bronze.dropna(
    subset=["product_id", "start_date", "usd_price"]
)

# Visualização de amostra e contagem final
display(df_importacao_bronze.limit(10))
print("Quantidade de registros em custos_importacao_bronze:", df_importacao_bronze.count())

In [0]:
# ======================================================================================
# ESCRITA - CUSTOS DE IMPORTAÇÃO BRONZE
# ======================================================================================

df_importacao_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.custos_importacao_bronze")

## 3. Transformação da tabela de vendas

Nesta etapa serão aplicados tratamentos técnicos mínimos:
- padronização de colunas
- ajuste de tipos numéricos
- conversão do campo de data
- preservação da granularidade transacional

In [0]:
# ======================================================================================
# LEITURA - VENDAS RAW
# ======================================================================================

df_vendas_raw = spark.table("lh_nautical.01_raw.vendas_raw")
print("Quantidade de registros em vendas_raw:", df_vendas_raw.count())
display(df_vendas_raw.limit(10))

In [0]:
# ======================================================================================
# INSPEÇÃO - VENDAS RAW
# ======================================================================================
# Esta célula ajuda a confirmar nomes e tipos antes da transformação.
# ======================================================================================

df_vendas_raw.printSchema()

In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - VENDAS BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - converter colunas de data com suporte a múltiplos formatos
# - limpar campos monetários antes da conversão numérica
# - converter ids, quantidades e valores
# - padronizar nomenclatura para modelo analítico
# - manter a granularidade original da venda
# ======================================================================================

df_vendas_bronze = standardize_dataframe_columns(df_vendas_raw)

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas de data (robusta)
# Suporta:
# - yyyy-MM-dd
# - dd/MM/yyyy
# - dd-MM-yyyy
# --------------------------------------------------------------------------------------
date_columns_candidates = ["sale_date", "date", "order_date", "data"]

for col_name in date_columns_candidates:
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.coalesce(
                F.expr(f"try_to_date({col_name}, 'yyyy-MM-dd')"),
                F.expr(f"try_to_date({col_name}, 'dd/MM/yyyy')"),
                F.expr(f"try_to_date({col_name}, 'dd-MM-yyyy')")
            )
        )

# --------------------------------------------------------------------------------------
# Limpeza de colunas monetárias antes do cast
# Remove símbolo monetário (R$), espaços e padroniza separador decimal
# --------------------------------------------------------------------------------------
monetary_columns = ["unit_price", "total"]

for col_name in monetary_columns:
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name).cast("string"), r"R\$\s*", "")
        )
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.regexp_replace(F.col(col_name), ",", ".")
        )

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas numéricas
# --------------------------------------------------------------------------------------
numeric_columns_candidates = {
    "id": IntegerType(),
    "sale_id": IntegerType(),
    "product_id": IntegerType(),
    "id_product": IntegerType(),
    "client_id": IntegerType(),
    "id_client": IntegerType(),
    "customer_id": IntegerType(),
    "quantity": IntegerType(),
    "qtd": IntegerType(),
    "qty": IntegerType(),
    "unit_price": DoubleType(),
    "total": DoubleType()
}

for col_name, col_type in numeric_columns_candidates.items():
    if col_name in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumn(
            col_name,
            F.col(col_name).cast(col_type)
        )

# --------------------------------------------------------------------------------------
# Padronização de nomes de colunas (ESSENCIAL PARA MODELAGEM)
# --------------------------------------------------------------------------------------
rename_mapping = {
    "id": "sale_id",
    "id_client": "customer_id",
    "client_id": "customer_id",
    "id_product": "product_id",
    "qtd": "quantity",
    "qty": "quantity",
    "total": "total_amount"
}

for old_col, new_col in rename_mapping.items():
    if old_col in df_vendas_bronze.columns:
        df_vendas_bronze = df_vendas_bronze.withColumnRenamed(old_col, new_col)

# --------------------------------------------------------------------------------------
# Visualização inicial para validação
# --------------------------------------------------------------------------------------
display(df_vendas_bronze.limit(10))

In [0]:
# ======================================================================================
# ESCRITA - VENDAS BRONZE
# ======================================================================================

df_vendas_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.vendas_bronze")

## 4. Transformação da tabela de clientes

Nesta etapa serão aplicados tratamentos técnicos mínimos:
- padronização de colunas
- ajuste de tipos
- preservação da estrutura principal da entidade cliente

In [0]:
# ======================================================================================
# LEITURA - CLIENTES RAW
# ======================================================================================

df_clientes_raw = spark.table("lh_nautical.01_raw.clientes_raw")
print("Quantidade de registros em clientes_raw:", df_clientes_raw.count())
display(df_clientes_raw.limit(10))

In [0]:
# ======================================================================================
# INSPEÇÃO - CLIENTES RAW
# ======================================================================================

df_clientes_raw.printSchema()

In [0]:
# ======================================================================================
# TRANSFORMAÇÃO - CLIENTES BRONZE
# ======================================================================================
# Objetivo:
# - padronizar nomes de colunas
# - renomear a chave de negócio para padrão analítico
# - higienizar campos textuais essenciais
# - corrigir e-mails com erro estrutural simples
# - remover duplicidades exatas
# - manter a integridade da entidade cliente
# ======================================================================================

df_clientes_bronze = standardize_dataframe_columns(df_clientes_raw)

# --------------------------------------------------------------------------------------
# Padronização de nomenclatura para modelo analítico
# --------------------------------------------------------------------------------------
rename_mapping = {
    "code": "customer_id",
    "full_name": "customer_name"
}

for old_col, new_col in rename_mapping.items():
    if old_col in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumnRenamed(old_col, new_col)

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas inteiras comuns
# --------------------------------------------------------------------------------------
client_integer_columns = ["customer_id", "client_id", "age"]

for col_name in client_integer_columns:
    if col_name in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumn(
            col_name,
            F.col(col_name).cast(IntegerType())
        )

# --------------------------------------------------------------------------------------
# Conversão condicional de colunas de data comuns
# --------------------------------------------------------------------------------------
client_date_columns = ["created_at", "signup_date", "birth_date", "date_of_birth"]

for col_name in client_date_columns:
    if col_name in df_clientes_bronze.columns:
        df_clientes_bronze = df_clientes_bronze.withColumn(
            col_name,
            F.coalesce(
                F.expr(f"try_to_date({col_name}, 'yyyy-MM-dd')"),
                F.expr(f"try_to_date({col_name}, 'dd/MM/yyyy')"),
                F.expr(f"try_to_date({col_name}, 'dd-MM-yyyy')")
            )
        )

# --------------------------------------------------------------------------------------
# Higienização de e-mail
# Corrige erro simples identificado na base: uso de "#" no lugar de "@"
# --------------------------------------------------------------------------------------
if "email" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )

    df_clientes_bronze = df_clientes_bronze.withColumn(
        "email",
        F.regexp_replace(F.col("email"), "#", "@")
    )

# --------------------------------------------------------------------------------------
# Higienização de nome do cliente
# --------------------------------------------------------------------------------------
if "customer_name" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "customer_name",
        F.trim(F.col("customer_name"))
    )

# --------------------------------------------------------------------------------------
# Higienização básica de localização
# Mantém o campo textual, mas remove excesso de espaços
# --------------------------------------------------------------------------------------
if "location" in df_clientes_bronze.columns:
    df_clientes_bronze = df_clientes_bronze.withColumn(
        "location",
        F.trim(F.regexp_replace(F.col("location"), r"\s+", " "))
    )

# --------------------------------------------------------------------------------------
# Remoção de duplicidades exatas
# --------------------------------------------------------------------------------------
total_clientes_antes = df_clientes_bronze.count()

df_clientes_bronze = df_clientes_bronze.dropDuplicates()

total_clientes_depois = df_clientes_bronze.count()
duplicados_removidos_clientes = total_clientes_antes - total_clientes_depois

print("Duplicados removidos em clientes:", duplicados_removidos_clientes)

# --------------------------------------------------------------------------------------
# Visualização inicial para validação
# --------------------------------------------------------------------------------------
display(df_clientes_bronze.limit(50))

In [0]:
display(
    df_clientes_bronze.filter(~F.col("email").contains("@"))
)

In [0]:
df_clientes_bronze.printSchema()

In [0]:
display(
    df_clientes_bronze.filter(F.col("customer_id").isNull())
)

In [0]:
# ======================================================================================
# ESCRITA - CLIENTES BRONZE
# ======================================================================================

df_clientes_bronze.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("lh_nautical.02_bronze.clientes_bronze")

## 5. Validações finais da camada bronze

Nesta etapa são verificadas:
- quantidade de registros por tabela
- persistência correta no catálogo
- disponibilidade para consumo posterior em dbt

In [0]:
# ======================================================================================
# VALIDAÇÃO FINAL - CONTAGENS
# ======================================================================================

spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.vendas_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.produtos_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.clientes_bronze").show()
spark.sql("SELECT COUNT(*) AS total FROM lh_nautical.02_bronze.custos_importacao_bronze").show()

In [0]:
# VALIDAÇÃO FINAL - AMOSTRAS DAS TABELAS BRONZE


tables = [
    "vendas_bronze",
    "produtos_bronze",
    "clientes_bronze",
    "custos_importacao_bronze"
]

for table in tables:
    print(f"\n🔹 TABELA: {table}")
    display(
        spark.table(f"lh_nautical.02_bronze.{table}").limit(5)
    )

## Conclusão

A camada `02_bronze` foi construída para as quatro fontes do projeto, preservando a rastreabilidade dos dados e preparando o ambiente para a camada `03_silver`.

### Próximos passos
- criar modelos `dbt` sobre as tabelas bronze
- consolidar entidades e regras de negócio na camada silver
- construir tabelas analíticas finais na camada gold para responder às questões 4 a 8